# CIT / Fouling Analysis — เดินเรื่องเองตั้งแต่ต้น (ฉบับส่วนตัว)

**หมายเหตุสำคัญ:** notebook นี้เป็นการวิเคราะห์อิสระที่เขียนสูตร/logic ใหม่ทั้งหมดด้วยตัวเอง
เพื่อให้เข้าใจทุกขั้นตอนได้จริง **ไม่ได้ import ฟังก์ชันจาก `src/`** และ **ไม่ได้อ่านผลลัพธ์ที่คำนวณไว้แล้ว**
จาก pipeline การผลิต (`notebooks/production/`) — ทุกอย่างเริ่มจากไฟล์ Excel ดิบ

- **Inputs:** `Process information data (2024-2026).xlsx`, `Curde Property.xlsx` (จาก `CPHT_DATA_DIR`)
- **Outputs:** ไม่มีการเขียนทับไฟล์ใดๆ ใน `Data/` หรือ `dashboard/data/` — เป็น notebook สำหรับสำรวจ/เรียนรู้เท่านั้น
  (ถ้าต้องการเก็บกราฟ จะเซฟไว้ที่ `figures/my_analysis/` แยกต่างหาก)
- **ขอบเขต:** ไม่แทนที่ pipeline การผลิตเดิม เป็นแค่การวิเคราะห์คู่ขนานเพื่อความเข้าใจของตัวเอง

## โครงเรื่อง 6 ขั้นตอน
1. ภาพรวมข้อมูล (HX sensors + crude property)
2. Clean ช่วง TAM และจัดการข้อมูลผิดปกติ
3. ใส่สูตร Q / ΔT / U-proxy / Fouling rate + วิเคราะห์ cleaning event
4. จัดอันดับ HX (ความสัมพันธ์กับ CIT, ความเสี่ยง fouling)
5. Feature engineering สำหรับพยากรณ์ CIT
6. เปรียบเทียบโมเดล + สำรวจแนวคิด network ระหว่าง HX

In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 140)
plt.rcParams['figure.figsize'] = (11, 4)

DATA_DIR = os.environ.get('CPHT_DATA_DIR', r'C:\Desktop\Bangchak Internship 2026\Data')
PROCESS_XLSX = os.path.join(DATA_DIR, 'Process information data (2024-2026).xlsx')
CRUDE_XLSX = os.path.join(DATA_DIR, 'Curde Property.xlsx')

FIG_DIR = os.path.join('figures', 'my_analysis')
os.makedirs(FIG_DIR, exist_ok=True)

### เกณฑ์การจำแนกค่า: MEASURED / CALCULATED / INFERRED / ASSUMED

โปรเจกต์นี้กำหนดไว้ว่าต้องแยกแยะ 4 ประเภทนี้เสมอ (ดู `CLAUDE.md`) — สรุปสั้นๆ สำหรับ notebook นี้:

| ประเภท | ตัวอย่างใน notebook นี้ |
|---|---|
| **MEASURED** | ค่าดิบจากเซนเซอร์ทุก tag ใน `proc_raw`/`crude_raw` (flow, temperature, API, SG, ...) |
| **CALCULATED** | Q, ΔT, U-proxy (`Q_norm`) — คำนวณตรงไปตรงมาจากค่า MEASURED ด้วยสูตรฟิสิกส์ที่ระบุไว้ |
| **INFERRED** | Fouling rate (Section 3), ผลลัพธ์ cleaning-event gain (Section 3) — มาจากการตีความรูปแบบข้อมูล ไม่ใช่วัดตรงๆ และมีความไม่แน่นอนสูงกว่า |
| **ASSUMED** | ค่าคงที่ในสูตร Cp/ρ ของ crude (Watson–Nelson correlation coefficients) — เป็นสูตรมาตรฐานที่ยังไม่ได้ยืนยันกับข้อมูล lab ของโรงกลั่นนี้โดยตรง |

**E101G ไม่มีเซนเซอร์และไม่ได้ถูกอนุมานใน notebook นี้เลย** (ตัดออกจาก `HX_TAGS` ทั้งหมด — ดู Section 1)

## Section 1 — ภาพรวมข้อมูล (Data Collection Overview)

โหลดข้อมูลดิบตรงจากไฟล์ Excel สองไฟล์:
- **Process data**: ค่าจากเซนเซอร์รายวัน (DCS historian) ของ HX ทุกตัว
- **Crude property**: คุณสมบัติน้ำมันดิบที่เข้าโรงกลั่น (API, SG, viscosity, MCRT, asphaltenes)

ไฟล์ Process data มี metadata อยู่บนสุด (แถว tag/unit/description) ก่อนจะถึงข้อมูลจริง
จึงต้อง `skiprows=6` เพื่อข้ามไปยังแถวที่เป็น header จริง (`Timestamp`, `{tag} - Average`, ...)

In [2]:
proc_raw = pd.read_excel(PROCESS_XLSX, skiprows=6)
proc_raw = proc_raw.drop(columns=[c for c in proc_raw.columns if str(c).startswith('Unnamed')])
proc_raw = proc_raw.rename(columns={c: c.replace(' - Average', '') for c in proc_raw.columns})
proc_raw['Timestamp'] = pd.to_datetime(proc_raw['Timestamp'])
proc_raw = proc_raw.set_index('Timestamp').sort_index()

print('process data:', proc_raw.shape, '|', proc_raw.index.min().date(), '->', proc_raw.index.max().date())
proc_raw.describe().T.head(10)

process data: (2008, 98) | 2021-01-01 -> 2026-07-01


,count,mean,std,min,25%,50%,75%,max
00FIC001.pv,2008.0,78.914885,24.761352,0.000000,62.871497,82.148995,95.875854,147.367449
00FC501.pv,2008.0,29.191603,14.806785,0.000000,19.432817,31.427482,41.927640,51.425453
1AI001.pv,2008.0,1.855058,0.801442,0.000000,1.505668,1.620298,1.899740,9.560200
1FC020.pv,2008.0,122.870597,15.054723,0.000000,121.580239,126.007117,130.203834,157.500000
1FC021.pv,2008.0,122.804384,17.589641,0.000000,121.985365,126.313608,130.311453,157.500000
1FC022.pv,2008.0,123.438424,15.699122,0.764952,122.057459,126.988803,130.668481,157.500000
1FC023.pv,2008.0,122.297236,16.263705,0.000000,121.657522,126.195557,130.153707,157.500000
1FC035.pv,2008.0,130.064072,30.307076,0.000000,119.999510,140.000383,150.735707,162.915982
1FC062.pv,2008.0,0.581496,4.965822,0.000000,0.000000,0.000000,0.000000,75.944882
1FI007.pv,2008.0,183.164140,26.980353,0.914488,168.213261,191.782871,202.063860,223.874720


In [3]:
# Curde Property.xlsx: header อยู่แถวที่ 3 (index 2), ข้อมูลจริงเริ่มถัดไป
# มีบาง row เป็น placeholder ว่าง (sheet เตรียมไว้ล่วงหน้าถึงปี 2026) ต้อง dropna ทิ้ง
crude_raw = pd.read_excel(CRUDE_XLSX, header=2)
crude_raw = crude_raw.drop(columns=[c for c in crude_raw.columns if str(c).startswith('Unnamed')])
crude_raw = crude_raw.rename(columns={'Curde Feed': 'Date'})
crude_raw['Date'] = pd.to_datetime(crude_raw['Date'], errors='coerce')
crude_raw = crude_raw.dropna(subset=['Date', 'API (none)']).set_index('Date').sort_index()
crude_raw = crude_raw[~crude_raw.index.duplicated(keep='last')]

print('crude property:', crude_raw.shape, '|', crude_raw.index.min().date(), '->', crude_raw.index.max().date())
crude_raw.describe().T

crude property: (1876, 6) | 2021-04-01 -> 2026-06-25


,count,unique,top,freq
API (none),1876.0,289.0,39.300,48.0
Specific gravity at 15.6C (none),1869.0,455.0,0.831,23.0
Viscosity (kinematic) at 50.0C (cSt),1876.0,397.0,2.980,24.0
Viscosity (kinematic) at 100.0C (cSt),1876.0,293.0,1.560,47.0
Carbon Residue (MCRT) (% wgt),1876.0,483.0,1.020,42.0
Asphaltenes (C7) (% wgt),1876.0,505.0,0.119,20.0


### รายการ tag ของแต่ละ HX (ระบุเองจากความรู้หน้างาน — ข้อเท็จจริงของ plant ไม่ใช่ logic ที่ยืมมา)

จำเป็นต้องรู้ว่า tag ไหนคือ cold-in / cold-out / flow ของ HX แต่ละตัว ถึงจะคำนวณอะไรต่อได้เลย
รายการนี้พิมพ์เองจากข้อมูล P&ID/tag list ของโรงงาน (ไม่ได้ import จากไฟล์ config เดิม)

**E101G ไม่มีเซนเซอร์วัดอุณหภูมิของตัวเอง** จึงตัดออกจากการคำนวณโดยตรง (ต้องอนุมานถ้าจำเป็น ซึ่งไม่ทำใน notebook นี้)

In [4]:
HX_TAGS = {
    'E101AB':  {'flow': '1FI007.pv', 'cold_in': '1TI102.pv', 'cold_out': '1TI101.pv'},
    'E101CD':  {'flow': '1FI008.pv', 'cold_in': '1TI102.pv', 'cold_out': '1TI104.pv'},
    'E101EF':  {'flow': '1FI009.pv', 'cold_in': '1TI102.pv', 'cold_out': '1TI109.pv'},
    'E102':    {'flow': '1fi005.pv', 'cold_in': '1TI107.pv', 'cold_out': '1TI106.pv'},
    'E103AB':  {'flow': '1FI015.pv', 'cold_in': '1TI225.pv', 'cold_out': '1TI136.pv'},
    'E104':    {'flow': '1FI015.pv', 'cold_in': '1TI136.pv', 'cold_out': '1TI112.pv'},
    'E105AB':  {'flow': '1FI015.pv', 'cold_in': '1TI112.pv', 'cold_out': '1TI114.pv'},
    'E106AB':  {'flow': '1FI016.pv', 'cold_in': '1TI225.pv', 'cold_out': '1TI128.pv'},
    'E107AB':  {'flow': '1FI016.pv', 'cold_in': '1TI128.pv', 'cold_out': '1TI130.pv'},
    'E108AB':  {'flow': '1FI016.pv', 'cold_in': '1TI130.pv', 'cold_out': '1TI132.pv'},
    'E109AB':  {'flow': '1FI016.pv', 'cold_in': '1TI132.pv', 'cold_out': '1TI134.pv'},
    'E110ABC': {'flow': '1FI017.pv', 'cold_in': '1TI225.pv', 'cold_out': '1TI124.pv'},
    'E111':    {'flow': '1FI017.pv', 'cold_in': '1TI124.pv', 'cold_out': '1TI123.pv'},
    'E112AB':  {'flow': '1FI017.pv', 'cold_in': '1TI123.pv', 'cold_out': '1TI126.pv'},
    'E113A':   {'flow': '1fi005.pv', 'cold_in': '1TI115.pv', 'cold_out': '1TI116.pv'},
}
CIT_TAG = '1TI116.pv'          # = E113A cold_out พอดี (จุดสุดท้ายก่อนเข้าเตา)
TOTAL_CHARGE_TAG = '1fi005.pv'

all_tags_needed = {CIT_TAG, TOTAL_CHARGE_TAG}
for cfg in HX_TAGS.values():
    all_tags_needed.update(cfg.values())
missing = [t for t in all_tags_needed if t not in proc_raw.columns]
print('tag ที่หาไม่เจอในไฟล์:', missing)

tag ที่หาไม่เจอในไฟล์: []


In [5]:
miss_pct = proc_raw[list(all_tags_needed)].isna().mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(proc_raw[list(all_tags_needed)].isna().astype(int).iloc[::7].T, cbar=False,
            cmap=['#eeeeee', '#d1495b'], ax=ax)
ax.set_title('Missing-value map (แดง = missing) — ทุก HX tag ที่ใช้จริง')
ax.set_xlabel('เวลา (sampled ทุก 7 วันเพื่อให้ดูง่าย)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_missing_map.png'), dpi=110)
plt.show()
print(miss_pct.describe())

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:7: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:7: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:7: UserWarning: Glyph 3621 (\N{THAI CHARACTER LO LING}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:7: UserWarning: Glyph 3634 (\N{THAI CHARACTER SARA AA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:7: UserWarning: Glyph 3607 (\N{THAI CHARACTER THO THAHAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:7: UserWarning: Glyph 3640 (\N{THAI CHARACTER SARA U}) miss

count    26.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
dtype: float64


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2423411211.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
proc_raw[TOTAL_CHARGE_TAG].plot(kind='hist', bins=40, ax=axes[0], title='Total charge distribution')
proc_raw[CIT_TAG].plot(kind='hist', bins=40, ax=axes[1], title='CIT distribution', color='orange')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_hist_charge_cit.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2884728127.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
proc_raw[TOTAL_CHARGE_TAG].plot(ax=axes[0], title='Total charge (1fi005.pv)')
proc_raw[CIT_TAG].plot(ax=axes[1], color='orange', title='CIT (1TI116.pv)')
proc_raw[[HX_TAGS['E105AB']['cold_in'], HX_TAGS['E105AB']['cold_out']]].plot(ax=axes[2],
    title='ตัวอย่าง: E105AB cold_in vs cold_out')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:6: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:6: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:6: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:6: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:6: UserWarning: Glyph 3618 (\N{THAI CHARACTER YO YAK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:6: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing fro

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: 

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '01_raw_timeseries_overview.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\363685852.py:7: 

In [8]:
crude_cols = crude_raw.columns.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, crude_cols):
    crude_raw[col].plot(ax=ax, title=col, lw=0.8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_crude_property_trends.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2961577864.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 2 — Clean ช่วง TAM และข้อมูลผิดปกติ

**TAM (Turnaround Maintenance)** คือช่วงที่โรงงานหยุดเดินเครื่องทั้งระบบ — flow รวมจะตกลงมาต่ำมาก
ใช้ `total charge flow` (rolling median 30 วัน) เป็นตัวจับสัญญาณ แล้ว plot threshold ให้เห็นตรงๆ
(ไม่ hardcode ตัวเลขแบบไม่มีที่มา — ปรับ `SHUTDOWN_THRESH` ได้จากกราฟด้านล่าง)

In [9]:
SHUTDOWN_THRESH = 300  # m3/hr โดยประมาณ — ปรับดูจากกราฟด้านล่างได้ ถ้า flow ปกติอยู่แถว 1500+

flow = proc_raw[TOTAL_CHARGE_TAG]
flow_roll = flow.rolling(30, center=True, min_periods=5).median()
TAM_MASK = flow_roll < SHUTDOWN_THRESH

fig, ax = plt.subplots()
flow.plot(ax=ax, alpha=0.4, label='daily flow')
flow_roll.plot(ax=ax, label='rolling median (30d)')
ax.axhline(SHUTDOWN_THRESH, color='red', ls='--', label=f'threshold = {SHUTDOWN_THRESH}')
ax.fill_between(flow.index, 0, flow.max(), where=TAM_MASK.reindex(flow.index, fill_value=False),
                 color='red', alpha=0.15, label='TAM window ที่ตรวจพบ')
ax.legend()
ax.set_title('Total charge flow + ช่วง TAM ที่ตรวจพบ')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_tam_detection.png'), dpi=110)
plt.show()

print('จำนวนวันที่ถูกจัดเป็น TAM:', int(TAM_MASK.sum()))

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:15: UserWarning: Glyph 3594 (\N{THAI CHARACTER CHO CHANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:15: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:15: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:15: UserWarning: Glyph 3591 (\N{THAI CHARACTER NGO NGU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:15: UserWarning: Glyph 3607 (\N{THAI CHARACTER THO THAHAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:15: UserWarning: Glyph 3637 (\N{THAI CHARACTER SARA 

จำนวนวันที่ถูกจัดเป็น TAM: 56


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:16: UserWarning: Glyph 3594 (\N{THAI CHARACTER CHO CHANG}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_tam_detection.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:16: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_tam_detection.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:16: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_tam_detection.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:16: UserWarning: Glyph 3591 (\N{THAI CHARACTER NGO NGU}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_tam_detection.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3531601485.py:16: UserWarning: Glyph 3607 (\N{THA

### แก้ข้อมูลผิดปกติ (outlier) ของอุณหภูมิฝั่งเย็น (cold-side)

ใช้ 2 เกณฑ์ร่วมกัน: (1) rolling z-score (ค่าที่เบี่ยงจากค่าเฉลี่ยรอบข้างมากผิดปกติ)
(2) ขอบเขตทางฟิสิกส์ (อุณหภูมิ crude ในช่วงนี้ไม่ควรต่ำกว่า 30°C หรือสูงกว่า 380°C)
จุดที่ถูก flag จะแทนด้วยค่าเฉลี่ยเคลื่อนที่ (rolling mean) รอบๆ จุดนั้น

In [10]:
def correct_outliers(s, win=30, z_thresh=3.0, lo=30, hi=380):
    roll_mean = s.rolling(win, center=True, min_periods=5).mean()
    roll_std = s.rolling(win, center=True, min_periods=5).std()
    z = (s - roll_mean) / roll_std.replace(0, np.nan)
    bad = (z.abs() > z_thresh) | (s < lo) | (s > hi)
    bad = bad.fillna(False)
    fixed = s.copy()
    fixed[bad] = roll_mean[bad]
    return fixed, bad

temp_tags = sorted({cfg['cold_in'] for cfg in HX_TAGS.values()} | {cfg['cold_out'] for cfg in HX_TAGS.values()})
proc_clean = proc_raw.copy()
flag_counts = {}
for tag in temp_tags:
    fixed, bad = correct_outliers(proc_raw[tag])
    proc_clean[tag] = fixed
    flag_counts[tag] = int(bad.sum())

pd.Series(flag_counts).sort_values(ascending=False).head(10)

1TI225.pv    806
1TI130.pv     25
1TI128.pv     18
1TI126.pv     16
1TI132.pv     15
1TI123.pv     10
1TI134.pv     10
1TI114.pv     10
1TI115.pv      8
1TI102.pv      8
dtype: int64

In [11]:
demo_tag = HX_TAGS['E105AB']['cold_out']
fig, ax = plt.subplots()
proc_raw[demo_tag].plot(ax=ax, label='raw', alpha=0.6)
proc_clean[demo_tag].plot(ax=ax, label='แก้ outlier แล้ว', lw=1.2)
ax.set_title(f'ตัวอย่างการแก้ outlier: {demo_tag} (E105AB cold_out)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_outlier_correction_example.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:7: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:7: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:7: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:7: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:7: UserWarning: Glyph 3618 (\N{THAI CHARACTER YO YAK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:7: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missi

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:8: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_outlier_correction_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:8: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_outlier_correction_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:8: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_outlier_correction_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3747404540.py:8: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '02_outlier_correction_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\

## Section 3 — ใส่สูตร Q / ΔT / U-proxy / Fouling rate + วิเคราะห์ cleaning event

### สูตรที่ใช้ (เขียนเองในนี้ อ้างอิงหลักวิศวกรรมมาตรฐาน ไม่ import จาก src/)

- **Cp ของ crude** (Watson–Nelson correlation, หน่วย kJ/kg·K): ใช้อุณหภูมิเฉลี่ยและ SG
- **ρ (density) ของ crude**: ปรับจาก SG ที่ 15.6°C ด้วยสัมประสิทธิ์ขยายตัวทางความร้อนโดยประมาณ
- **Q (duty, kW)** = ρ × flow × Cp × ΔT_cold / 3600
- **Q_norm** = Q / total charge — ใช้เป็น proxy แทน U จริง เพราะไม่มีข้อมูลพื้นที่ผิว (Area) และ LMTD ในไฟล์ดิบชุดนี้
  (ถ้าต้องการ U จริงต้องยืนยันว่ามีข้อมูล Area ของแต่ละ HX ก่อน — ตอนนี้ใช้ Q_norm แทนไปก่อน)

In [12]:
def cp_crude_kJ_per_kgK(T_C, SG):
    T_F = T_C * 9 / 5 + 32
    cp_btu_per_lb_F = (0.388 + 0.00045 * T_F) / np.sqrt(SG)
    return cp_btu_per_lb_F * 4.1868

def rho_crude_kg_per_m3(T_C, SG):
    rho_15_6 = SG * 999.016
    alpha = 613.9723 / rho_15_6 ** 2
    return rho_15_6 * (1 - alpha * (T_C - 15.6))

sg_daily = crude_raw['Specific gravity at 15.6C (none)'].reindex(
    proc_clean.index.union(crude_raw.index)).sort_index().ffill().reindex(proc_clean.index)

cit = proc_clean[CIT_TAG].copy()
cit[TAM_MASK.reindex(cit.index, fill_value=False)] = np.nan

hx_results = {}
for hx, cfg in HX_TAGS.items():
    cold_in = proc_clean[cfg['cold_in']]
    cold_out = proc_clean[cfg['cold_out']]
    flow_hx = proc_clean[cfg['flow']]
    t_avg = (cold_in + cold_out) / 2
    cp = cp_crude_kJ_per_kgK(t_avg, sg_daily)
    rho = rho_crude_kg_per_m3(t_avg, sg_daily)
    dT = cold_out - cold_in
    Q_kW = rho * flow_hx * cp * dT / 3600
    Q_norm = Q_kW / proc_clean[TOTAL_CHARGE_TAG]
    df_hx = pd.DataFrame({'cold_in': cold_in, 'cold_out': cold_out, 'dT': dT,
                          'Q_kW': Q_kW, 'Q_norm': Q_norm})
    df_hx.loc[TAM_MASK.reindex(df_hx.index, fill_value=False), :] = np.nan
    hx_results[hx] = df_hx

hx_results['E105AB'].describe()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\29499592.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  proc_clean.index.union(crude_raw.index)).sort_index().ffill().reindex(proc_clean.index)


,cold_in,cold_out,dT,Q_kW,Q_norm
count,1952.000000,1952.000000,1952.000000,1890.000000,1890.000000
mean,168.717594,230.534006,61.816412,6716.484801,12.840569
std,6.705716,8.179454,6.215481,1050.119021,1.775402
min,139.488441,185.652515,27.135258,2223.131022,5.756497
25%,165.306346,224.982978,57.096574,5931.775042,11.393926
50%,168.874108,229.572919,62.392335,6759.802069,12.894137
75%,172.090510,234.917831,66.485121,7553.431755,14.222411
max,195.867855,259.564665,80.217387,9165.346955,21.094594


In [13]:
hx_list = list(HX_TAGS.keys())
fig, axes = plt.subplots(5, 3, figsize=(15, 16), sharex=True)
for ax, hx in zip(axes.flat, hx_list):
    q = hx_results[hx]['Q_norm']
    ax2 = ax.twinx()
    q.plot(ax=ax, color='tab:blue', lw=0.8)
    cit.plot(ax=ax2, color='tab:orange', lw=0.6, alpha=0.6)
    ax.set_title(hx, fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes.flat[len(hx_list):]:
    ax.axis('off')
fig.suptitle('Q_norm (น้ำเงิน, แกนซ้าย) เทียบกับ CIT (ส้ม, แกนขวา) ต่อ HX', y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:13: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:13: UserWarning: Glyph 3657 (\N{THAI CHARACTER MAI THO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:13: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:13: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:13: UserWarning: Glyph 3591 (\N{THAI CHARACTER NGO NGU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:13: UserWarning: Glyph 3636 (\N{THAI CHARACTER SARA I}) mis

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3657 (\N{THAI CHARACTER MAI THO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3591 (\N{THAI CH

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3657 (\N{THAI CHARACTER MAI THO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3591 (\N{THAI CH

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3657 (\N{THAI CHARACTER MAI THO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_q_vs_cit_grid.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4226200749.py:14: UserWarning: Glyph 3591 (\N{THAI CH

### Fouling rate: ทำ linear fit ของ Q_norm ในแต่ละ "run" (ช่วงต่อเนื่องที่ไม่ใช่ TAM)

วิธีง่ายที่สุด: ตัดข้อมูลเป็นช่วงๆ ตาม TAM window แล้ว fit เส้นตรงใน run ที่มีข้อมูลพอ (≥20 จุด)
slope ที่ได้ (Q_norm ต่อวัน) คือ fouling rate อย่างหยาบ — ถ้าติดลบแปลว่า duty กำลังลดลงตามเวลา (fouling)

In [14]:
def detect_runs(index, tam_mask):
    valid = ~tam_mask.reindex(index).fillna(True)
    runs, start, prev = [], None, None
    for ts, ok in valid.items():
        if ok and start is None:
            start = ts
        if not ok and start is not None:
            runs.append((start, prev))
            start = None
        prev = ts
    if start is not None:
        runs.append((start, prev))
    return runs

fouling_rates = {}
fouling_detail = {}
for hx, df_hx in hx_results.items():
    runs = detect_runs(df_hx.index, TAM_MASK)
    slopes = []
    for (s, e) in runs:
        seg = df_hx.loc[s:e, 'Q_norm'].dropna()
        if len(seg) < 20 or (seg.index[-1] - seg.index[0]).days < 20:
            continue
        x = (seg.index - seg.index[0]).days.values.astype(float)
        slope = np.polyfit(x, seg.values, 1)[0]
        slopes.append(slope)
    fouling_detail[hx] = slopes
    fouling_rates[hx] = np.nanmedian(slopes) if slopes else np.nan

fr_series = pd.Series(fouling_rates).sort_values()
fr_series

E113A     -0.015469
E110ABC   -0.010481
E103AB    -0.007587
E101EF    -0.005612
E106AB    -0.005286
E101AB    -0.004750
E101CD    -0.003921
E108AB    -0.003427
E104      -0.001960
E102      -0.001237
E105AB    -0.000468
E111       0.000348
E107AB     0.001340
E112AB     0.001501
E109AB     0.001613
dtype: float64

**⚠️ ข้อควรระวัง (INFERRED, ไม่ใช่ตัวเลขที่ใช้งานจริงของ pipeline):** slope ด้านบนได้จาก `np.polyfit`
(OLS ธรรมดา) บน `Q_norm` ดิบทั้ง run โดยไม่ได้กรอง in-service state และไม่ได้ใช้ robust estimator —
เป็นวิธีง่ายสุดสำหรับหัดทำความเข้าใจเท่านั้น pipeline การผลิตจริง (`pipeline/compute_fouling_rate.py`)
เจอว่าวิธีนี้ไม่น่าเชื่อถือพอ จึงเปลี่ยนไปใช้ **Theil-Sen slope + in-service mask + reliability gate**
แทน (ผ่านเกณฑ์แค่ 69/125 HX-run) ตัวเลขในตาราง `fr_series` ด้านบนจึงอาจไม่ตรงกับ `Data/Fouling_Rate_By_Run.csv`
ที่เป็นตัวจริง — ใช้ตารางนี้เพื่อดู "ทิศทาง" คร่าวๆ เท่านั้น ไม่ใช่ตัวเลขอ้างอิง

In [15]:
demo_hx = 'E113A'
seg_runs = detect_runs(hx_results[demo_hx].index, TAM_MASK)
fig, ax = plt.subplots()
hx_results[demo_hx]['Q_norm'].plot(ax=ax, lw=0.8, label='Q_norm')
for i, (s, e) in enumerate(seg_runs):
    seg = hx_results[demo_hx].loc[s:e, 'Q_norm'].dropna()
    if len(seg) < 20:
        continue
    x = (seg.index - seg.index[0]).days.values.astype(float)
    slope, intercept = np.polyfit(x, seg.values, 1)
    ax.plot(seg.index, intercept + slope * x, color='red', lw=1.5,
             label='linear fit ต่อ run' if i == 0 else None)
ax.legend()
ax.set_title(f'{demo_hx}: Q_norm ต่อ run พร้อมเส้น fouling-rate fit')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_fouling_rate_fit_example.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:11: UserWarning: This axis already has a converter set and is updating to a potentially incompatible converter
  ax.plot(seg.index, intercept + slope * x, color='red', lw=1.5,
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:15: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:15: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:15: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:15: UserWarning: Glyph 3614 (\N{THAI CHARACTER PHO PHAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\302640405

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:16: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_fouling_rate_fit_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:16: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_fouling_rate_fit_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:16: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_fouling_rate_fit_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3026404057.py:16: UserWarning: Glyph 3614 (\N{THAI CHARACTER PHO PHAN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_fouling_rate_fit_example.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\302640405

### ตรวจจับ cleaning event (การกระโดดขึ้นของ Q_norm) แล้วดูว่า Q/CIT เพิ่มขึ้นแค่ไหนหลัง cleaning

วิธี: หา rolling median ของ Q_norm แล้วดู diff ในช่วง 5 วัน ถ้ากระโดดขึ้นเกิน threshold (สัดส่วนของค่ามัธยฐาน)
ถือว่าเป็น event — เทียบ median ของ Q และ CIT ในหน้าต่าง ±10 วัน (เว้นช่วง gap 2 วัน) ก่อน/หลัง event

In [16]:
def detect_events(q_norm, smooth_win=5, jump_frac=0.15):
    smooth = q_norm.rolling(smooth_win, min_periods=2).median()
    jump = smooth.diff(smooth_win)
    thresh = jump_frac * q_norm.median()
    is_event = jump > thresh
    event_dates, last = [], None
    for d in is_event[is_event].index:
        if last is None or (d - last).days > 10:
            event_dates.append(d)
        last = d
    return event_dates, thresh

PRE, POST, GAP = 10, 10, 2
event_tables = {}
for hx, df_hx in hx_results.items():
    events, thresh = detect_events(df_hx['Q_norm'])
    rows = []
    for ev in events:
        q = df_hx['Q_norm']
        before = q.loc[ev - pd.Timedelta(days=PRE + GAP): ev - pd.Timedelta(days=GAP)].median()
        after = q.loc[ev + pd.Timedelta(days=GAP): ev + pd.Timedelta(days=GAP + POST)].median()
        cb = cit.loc[ev - pd.Timedelta(days=PRE + GAP): ev - pd.Timedelta(days=GAP)].median()
        ca = cit.loc[ev + pd.Timedelta(days=GAP): ev + pd.Timedelta(days=GAP + POST)].median()
        rows.append({'date': ev, 'Q_before': before, 'Q_after': after,
                      'Q_gain_pct': 100 * (after - before) / before if before else np.nan,
                      'CIT_before': cb, 'CIT_after': ca, 'CIT_gain_C': ca - cb})
    event_tables[hx] = pd.DataFrame(rows)

event_tables['E105AB']

,date,Q_before,Q_after,Q_gain_pct,CIT_before,CIT_after,CIT_gain_C
0,2021-05-12,9.477838,11.771665,24.202005,252.088520,253.713989,1.625468
1,2021-05-31,10.139884,12.060989,18.946032,252.463768,254.192764,1.728996
2,2021-06-25,11.334288,13.774693,21.531172,255.184051,254.549758,-0.634293
3,2021-07-23,13.150352,12.918716,-1.761447,250.749398,251.745461,0.996063
4,2021-12-24,12.416204,10.605631,-14.582337,247.741176,246.900957,-0.840219
5,2022-02-23,12.051203,12.840854,6.552469,248.185805,249.705021,1.519216
6,2022-08-11,12.735660,13.245700,4.004818,244.315138,248.145344,3.830206
7,2024-06-08,7.712466,14.601957,89.329301,249.405018,263.102770,13.697752
8,2024-08-10,15.065329,13.505775,-10.351943,259.307902,260.826808,1.518906
9,2024-09-15,13.582999,14.883082,9.571403,257.803880,258.515467,0.711586


**⚠️ ข้อควรระวัง (INFERRED, ยังไม่ยืนยัน — ไม่ใช่ cleaning event ที่คอนเฟิร์มแล้ว):** `detect_events`
ด้านบนจับ "การกระโดดขึ้นของ Q_norm เกิน 15% ของค่ามัธยฐาน" เฉยๆ ไม่มีการ cross-check กับ config
การสลับ shell หรือบันทึกการล้างจริง ดังนั้นอาจปนกับเหตุการณ์อื่นที่ไม่ใช่การล้าง (เช่น crude เปลี่ยนสภาพ,
เซนเซอร์กลับมาทำงานปกติ) pipeline การผลิตจริง (`pipeline/export_cleaning_history.py`) เชื่อ event
เฉพาะที่มี config ยืนยันการสลับ shell ประกอบเท่านั้น และ `notebooks/production/06_cleaning_event_validation.ipynb`
มีอยู่เพราะ event ที่ตรวจจับแบบดิบต้องให้วิศวกรยืนยันก่อนเชื่อได้ — ตัวเลข `gain_summary` ด้านล่างจึงเป็นแค่
"ผู้สมัคร" (candidate) ที่ยังไม่ยืนยัน ไม่ใช่ผลการล้างที่คอนเฟิร์มแล้ว

In [17]:
gain_summary = pd.DataFrame({
    hx: [df['Q_gain_pct'].median() if len(df) else np.nan,
         df['CIT_gain_C'].median() if len(df) else np.nan,
         len(df)]
    for hx, df in event_tables.items()
}, index=['median_Q_gain_pct', 'median_CIT_gain_C', 'n_events']).T.sort_values('median_Q_gain_pct')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
gain_summary['median_Q_gain_pct'].plot(kind='barh', ax=axes[0], title='Median Q gain (%) หลัง cleaning event')
gain_summary['median_CIT_gain_C'].plot(kind='barh', ax=axes[1], color='orange',
                                        title='Median CIT gain (°C) หลัง cleaning event')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_before_after_cleaning_gain.png'), dpi=110)
plt.show()
gain_summary

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2625498522.py:12: UserWarning: Glyph 3627 (\N{THAI CHARACTER HO HIP}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2625498522.py:12: UserWarning: Glyph 3621 (\N{THAI CHARACTER LO LING}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2625498522.py:12: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2625498522.py:12: UserWarning: Glyph 3591 (\N{THAI CHARACTER NGO NGU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2625498522.py:13: UserWarning: Glyph 3627 (\N{THAI CHARACTER HO HIP}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '03_before_after_cleaning_gain.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\26254

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2625498522.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,median_Q_gain_pct,median_CIT_gain_C,n_events
E106AB,3.819176,-0.540914,18.0
E101CD,4.612030,0.126598,2.0
E110ABC,4.941304,-0.378060,20.0
E101AB,6.652193,0.336691,1.0
E103AB,6.718948,0.192438,12.0
E104,6.902694,-2.449855,6.0
E105AB,8.437905,1.572342,18.0
E102,9.186688,-0.171742,28.0
E101EF,11.366827,1.534889,2.0
E109AB,14.056361,0.232541,48.0


## Section 4 — จัดอันดับ HX

จัดอันดับด้วย 2 เกณฑ์แยกกันชัดๆ (ไม่รวมเป็นสูตร black-box ซับซ้อน เพื่อให้ตีความง่าย):
1. **ความสัมพันธ์ Q กับ CIT** (Pearson correlation) — HX ไหนมีผลต่อ CIT มากที่สุด
2. **ขนาด fouling rate** (จาก Section 3) — HX ไหนเสื่อมเร็วที่สุด

In [18]:
corrs = {}
for hx, df_hx in hx_results.items():
    both = pd.concat([df_hx['Q_norm'], cit], axis=1).dropna()
    corrs[hx] = both.corr().iloc[0, 1] if len(both) > 30 else np.nan
corr_series = pd.Series(corrs).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
corr_series.plot(kind='barh', ax=ax, title='Q_norm–CIT correlation ต่อ HX')
ax.axvline(0, color='grey', lw=0.8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_qcit_correlation_ranking.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1243883873.py:10: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1243883873.py:10: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1243883873.py:10: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1243883873.py:11: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '04_qcit_correlation_ranking.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1243883873.py:11: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '04_qcit_correlation_ranking.png'), dpi=110)
C:\Use

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1243883873.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
fig, ax = plt.subplots(figsize=(8, 6))
fr_series.plot(kind='barh', ax=ax, color='firebrick', title='Fouling rate (median run slope, Q_norm/วัน) ต่อ HX')
ax.axvline(0, color='grey', lw=0.8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_fouling_rate_ranking.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:4: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:4: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:4: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:4: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:4: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:4: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missin

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:5: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '04_fouling_rate_ranking.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:5: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '04_fouling_rate_ranking.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:5: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '04_fouling_rate_ranking.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:5: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '04_fouling_rate_ranking.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:5: UserWar

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\4288541596.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
rank_df = pd.DataFrame({'qcit_corr': corr_series, 'fouling_rate': fr_series}).dropna()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(rank_df['qcit_corr'], rank_df['fouling_rate'])
for hx, row in rank_df.iterrows():
    ax.annotate(hx, (row['qcit_corr'], row['fouling_rate']), fontsize=8)
ax.axhline(0, color='grey', lw=0.5)
ax.axvline(0, color='grey', lw=0.5)
ax.set_xlabel('Q_norm–CIT correlation (สูง = มีผลต่อ CIT มาก)')
ax.set_ylabel('Fouling rate (ติดลบมาก = เสื่อมเร็ว)')
ax.set_title('HX ที่อยู่มุมล่างขวา = มีผลต่อ CIT มาก + เสื่อมเร็ว (ควรจับตาก่อน)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_combined_risk_scatter.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:11: UserWarning: Glyph 3626 (\N{THAI CHARACTER SO SUA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:11: UserWarning: Glyph 3641 (\N{THAI CHARACTER SARA UU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:11: UserWarning: Glyph 3591 (\N{THAI CHARACTER NGO NGU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:11: UserWarning: Glyph 3617 (\N{THAI CHARACTER MO MA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:11: UserWarning: Glyph 3637 (\N{THAI CHARACTER SARA II}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:11: UserWarning: Glyph 3612 (\N{THAI CHARACTER PHO PHUNG}) 

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3264227888.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### รวม 2 เกณฑ์เป็นตารางเดียว (quick view เท่านั้น — ไม่ใช่ risk score ที่ใช้งานจริง)

Rank-normalize ทั้งสองแกน (0 = แย่สุด, 1 = ดีสุด) แล้วรวมง่ายๆ เพื่อดูภาพรวมเร็วๆ ว่า HX ไหน
"มีผลต่อ CIT มาก + เสื่อมเร็ว" พร้อมกัน — pipeline การผลิตจริงมี risk score ที่ละเอียดกว่านี้มาก
(`Engineering_Priority_Score.csv` จาก `08_cleaning_priority.ipynb`, รวม safety weight, effort tier,
trajectory ด้วย) ตารางนี้เป็นแค่มุมมองเร็วๆ ของตัวเอง ไม่ควรใช้แทนอันนั้น

In [21]:
rank_df['impact_rank'] = rank_df['qcit_corr'].abs().rank(pct=True)
rank_df['fouling_severity_rank'] = (-rank_df['fouling_rate']).rank(pct=True)  # ยิ่งลบมาก (เสื่อมเร็ว) ยิ่งได้ rank สูง
rank_df['combined_quick_score'] = (rank_df['impact_rank'] + rank_df['fouling_severity_rank']) / 2
rank_df.sort_values('combined_quick_score', ascending=False)[
    ['qcit_corr', 'fouling_rate', 'impact_rank', 'fouling_severity_rank', 'combined_quick_score']]

,qcit_corr,fouling_rate,impact_rank,fouling_severity_rank,combined_quick_score
E110ABC,0.247054,-0.010481,0.666667,0.933333,0.800000
E101CD,0.390345,-0.003921,0.866667,0.600000,0.733333
E101AB,0.313572,-0.004750,0.733333,0.666667,0.700000
E102,0.483210,-0.001237,1.000000,0.400000,0.700000
E106AB,0.178077,-0.005286,0.600000,0.733333,0.666667
E103AB,0.127494,-0.007587,0.400000,0.866667,0.633333
E105AB,0.472250,-0.000468,0.933333,0.333333,0.633333
E113A,0.108181,-0.015469,0.266667,1.000000,0.633333
E104,0.136385,-0.001960,0.466667,0.466667,0.466667
E101EF,0.021938,-0.005612,0.066667,0.800000,0.433333


## Section 5 — Network Feature Engineering for CIT

ส่วนนี้ต้องการอธิบายผลของ HX network ต่อ CIT จึง **ไม่ใช้ `CIT_lag1`, `CIT_lag7` หรือ `CIT_roll7` เป็น model feature**
Q ถูกรวมตาม Branch A/B/C ตาม topology จริง และเพิ่ม inlet temperature, total charge, crude property และ operating state เป็น control variables
Same-day Q ของ E113A/E112C ถูกตัดออก เพราะใช้ CIT ในสูตร Q เอง จึงเป็น target leakage

In [22]:
# Leakage-safe network feature table. CIT history is intentionally NOT a model feature.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'domain' / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.domain.config import DATA_DIR as DATA_CANON

def _read_ts(name):
    return pd.read_csv(Path(DATA_CANON) / name, parse_dates=['Timestamp']).set_index('Timestamp').sort_index()

model_proc = _read_ts('Process_information_with_crude.csv')
model_q = _read_ts('Feature_Q.csv')
model_state = _read_ts('Operating_State.csv')
model_idx = model_proc.index.intersection(model_q.index).intersection(model_state.index)
model_proc, model_q, model_state = [x.reindex(model_idx) for x in (model_proc, model_q, model_state)]

NETWORK_BRANCHES = {
    'Q_Branch_A': ['E103AB', 'E104', 'E105AB'],
    'Q_Branch_B': ['E106AB', 'E107AB', 'E108AB', 'E109AB'],
    'Q_Branch_C': ['E110ABC', 'E111', 'E112AB'],
}
feat = pd.DataFrame(index=model_idx)
for name, hxs in NETWORK_BRANCHES.items():
    feat[name] = model_q[hxs].sum(axis=1, min_count=len(hxs))

# Controls are required to stop Q coefficients absorbing inlet/throughput/crude changes.
feat['T_CPHT2_in'] = model_proc['1TI225.pv']
feat['total_charge'] = model_proc['1fi005.pv']
for col in ['SG_15_6C', 'Visc_50C_cSt', 'Visc_100C_cSt', 'MCRT_pct', 'Asphaltenes_pct']:
    feat[col] = model_proc[col]
feat['E112C_active'] = model_state['E112C'].eq('SUBSTITUTE_ACTIVE').astype(float)
feat['CIT'] = model_proc['1TI116.pv']

# Exclude unresolved final-shell state conflicts and low-charge rows.
final_conflict = (model_state['E113A'].isin(['NORMAL', 'SUBSTITUTE_ACTIVE']) ==
                  model_state['E112C'].isin(['NORMAL', 'SUBSTITUTE_ACTIVE']))
feat = feat[(model_proc['1fi005.pv'] >= 200) & ~final_conflict]
feat_clean = feat.replace([np.inf, -np.inf], np.nan).dropna()

q_feature_cols = list(NETWORK_BRANCHES)
control_cols = ['T_CPHT2_in', 'total_charge', 'SG_15_6C', 'Visc_50C_cSt',
                'Visc_100C_cSt', 'MCRT_pct', 'Asphaltenes_pct', 'E112C_active']
feature_cols = q_feature_cols + control_cols
assert not any(c.startswith('CIT_lag') or c.startswith('CIT_roll') for c in feature_cols)
assert 'E113A' not in ' '.join(feature_cols) and 'E112C_Q' not in feature_cols
print('network model table:', feat_clean.shape, '| Q:', q_feature_cols, '| controls:', control_cols)
corr_vs_cit = feat_clean[feature_cols + ['CIT']].corr()['CIT'].drop('CIT').sort_values()
corr_vs_cit

network model table: (1637, 12) | Q: ['Q_Branch_A', 'Q_Branch_B', 'Q_Branch_C'] | controls: ['T_CPHT2_in', 'total_charge', 'SG_15_6C', 'Visc_50C_cSt', 'Visc_100C_cSt', 'MCRT_pct', 'Asphaltenes_pct', 'E112C_active']


total_charge      -0.197792
T_CPHT2_in        -0.192544
SG_15_6C          -0.062670
Asphaltenes_pct   -0.048363
MCRT_pct          -0.015801
Q_Branch_B         0.037244
E112C_active       0.093737
Visc_100C_cSt      0.118080
Visc_50C_cSt       0.131413
Q_Branch_C         0.326173
Q_Branch_A         0.404659
Name: CIT, dtype: float64

In [23]:
fig, ax = plt.subplots(figsize=(7, 10))
corr_vs_cit.plot(kind='barh', ax=ax, title='Correlation ของแต่ละ feature กับ CIT')
ax.axvline(0, color='grey', lw=0.7)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_feature_correlation_ranking.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:4: UserWarning: Glyph 3586 (\N{THAI CHARACTER KHO KHAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:4: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:4: UserWarning: Glyph 3591 (\N{THAI CHARACTER NGO NGU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:4: UserWarning: Glyph 3649 (\N{THAI CHARACTER SARA AE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:4: UserWarning: Glyph 3605 (\N{THAI CHARACTER TO TAO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:4: UserWarning: Glyph 3656 (\N{THAI CHARACTER MAI EK}) missing 

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3557301129.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
top_feats = corr_vs_cit.abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, f in zip(axes.flat, top_feats):
    ax.scatter(feat_clean[f], feat_clean['CIT'], s=5, alpha=0.4)
    ax.set_title(f'{f} vs CIT (r={corr_vs_cit[f]:.2f})', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_top_feature_scatter.png'), dpi=110)
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(feat_clean[top_feats + ['CIT']].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_correlation_heatmap.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\621543644.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\621543644.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6 — เปรียบเทียบโมเดล + สำรวจแนวคิด Network

### 6.1 CIT network explanatory model — เทียบกับ persistence benchmark

**สำคัญ:** แบ่ง train/test แบบ **เรียงตามเวลา** (train = ช่วงต้น, test = ช่วงหลัง) ไม่สุ่มแบ่ง
เพราะข้อมูล time-series ถ้าสุ่มแบ่งจะมีการ "แอบดูอนาคต" (leakage) — และต้องเทียบกับ baseline ง่ายๆ เสมอ
ถึงจะรู้ว่าโมเดลที่ซับซ้อนกว่าคุ้มค่าจริงหรือแค่ดูดีเฉยๆ

In [25]:
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Chronological split. Persistence is an external benchmark, never a model feature.
split = int(len(feat_clean) * 0.8)
train, test = feat_clean.iloc[:split], feat_clean.iloc[split:]
y_train, y_test = train['CIT'], test['CIT']

def fit_ridge(cols):
    scaler = StandardScaler().fit(train[cols])
    model = RidgeCV(alphas=np.logspace(-3, 3, 31), cv=TimeSeriesSplit(5))
    model.fit(scaler.transform(train[cols]), y_train)
    pred = model.predict(scaler.transform(test[cols]))
    coef_native = pd.Series(model.coef_ / scaler.scale_, index=cols)
    coef_std = pd.Series(model.coef_, index=cols)
    return model, scaler, pred, coef_native, coef_std

controls_model, controls_scaler, pred_controls, _, _ = fit_ridge(control_cols)
network_model, network_scaler, pred_network, network_coef_native, network_coef_std = fit_ridge(feature_cols)
persistence = feat_clean['CIT'].shift(1).reindex(test.index)

preds = {
    'Persistence benchmark': persistence,
    'Controls only': pd.Series(pred_controls, index=test.index),
    'Branch Q + controls': pd.Series(pred_network, index=test.index),
}
metrics = pd.DataFrame({
    name: [r2_score(y_test[p.notna()], p.dropna()),
           np.sqrt(mean_squared_error(y_test[p.notna()], p.dropna())),
           mean_absolute_error(y_test[p.notna()], p.dropna())]
    for name, p in preds.items()
}, index=['R2', 'RMSE', 'MAE']).T
metrics['delta_RMSE_vs_controls'] = metrics.loc['Controls only', 'RMSE'] - metrics['RMSE']
metrics.sort_values('RMSE')

,R2,RMSE,MAE,delta_RMSE_vs_controls
Persistence benchmark,0.861688,1.054207,0.710412,2.320516
Controls only,-0.417378,3.374723,2.604710,0.000000
Branch Q + controls,-0.529766,3.505967,2.801768,-0.131243


In [26]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
metrics['R2'].plot(kind='bar', ax=axes[0], title='Chronological test R2')
axes[0].axhline(0, color='grey', lw=0.8)
metrics['RMSE'].plot(kind='bar', ax=axes[1], color='firebrick', title='Chronological test RMSE')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_model_comparison.png'), dpi=110)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
y_test.plot(ax=ax, label='Measured CIT', lw=1)
preds['Branch Q + controls'].plot(ax=ax, label='Branch Q + controls', lw=1)
preds['Controls only'].plot(ax=ax, label='Controls only', lw=.9, alpha=.8)
ax.legend(); ax.set_title('CIT explanatory model — no CIT-history features')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_actual_vs_predicted.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3868822589.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\3868822589.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**วิธีตีความ:** Persistence ถูกเก็บไว้เป็น benchmark เท่านั้น ไม่ได้ป้อนเข้าโมเดล
ถ้า `Branch Q + controls` ไม่ดีกว่า `Controls only` บน chronological holdout แสดงว่า Q-level relationship ยังไม่คงที่ข้าม operating regime; ห้ามสรุปว่า coefficient เป็น causal effect
สำหรับ engineering sensitivity ให้เน้น first-difference model ใน Section 7.6: `dCIT ~ dQ_A + dQ_B + dQ_C + controls`

In [27]:
coef_table = pd.DataFrame({
    'coefficient_native_units': network_coef_native,
    'effect_per_1SD': network_coef_std,
}).sort_values('effect_per_1SD', key=abs, ascending=False)
display(coef_table.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
network_coef_std.sort_values().plot.barh(ax=axes[0])
axes[0].axvline(0, color='black', lw=.8); axes[0].set_title('Ridge effect per 1 SD')
axes[0].set_xlabel('effect on same-day CIT (degC)')
network_coef_native[q_feature_cols].sort_values().plot.barh(ax=axes[1], color='tab:blue')
axes[1].axvline(0, color='black', lw=.8); axes[1].set_title('Branch-Q coefficients in native units')
axes[1].set_xlabel('degC per [kW/(m3/h)]')
fig.savefig(os.path.join(FIG_DIR, '06_feature_importance.png'), dpi=110, bbox_inches='tight')
plt.show()

,coefficient_native_units,effect_per_1SD
Q_Branch_A,0.5936,3.2935
Q_Branch_C,0.5447,2.4011
Q_Branch_B,-0.4622,-1.8130
T_CPHT2_in,0.0701,1.8053
Visc_50C_cSt,2.3634,1.0032
MCRT_pct,-2.8917,-0.9154
Asphaltenes_pct,-8.1312,-0.6372
E112C_active,1.6213,0.4590
SG_15_6C,-48.6195,-0.3606
total_charge,-0.0092,-0.2767


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2060777131.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.2 แนวคิด Network — HX เชื่อมกันเป็นสายการไหลของ crude

ตามลำดับการไหลจริงของ crude (ความรู้หน้างาน ไม่ใช่โมเดลที่ fit จากข้อมูล):
CPHT-1 (ก่อนเข้า desalter): `E101AB/CD/EF -> E102`
CPHT-2 (ก่อนเข้าเตา): `E103AB -> E104 -> E105AB` และอีกสาย `E106AB -> E107AB -> E108AB -> E109AB` และ `E110ABC -> E111 -> E112AB -> E113A (CIT)`

**ข้อควรระวังที่ต้องพูดตรงๆ:** นี่เป็นแค่ลำดับตำแหน่งทางกายภาพ (topology) ไม่ใช่โมเดลจำลองการไหลความร้อนจริง (thermal cascade)
เพราะไม่มีข้อมูล Area/LMTD ของแต่ละ HX ที่ยืนยันแล้วในชุดข้อมูลนี้ การจะสร้างโมเดล cascade ที่แม่นยำ
ต้องมีข้อมูล mechanical datasheet เพิ่มเติมและการ validate กับพนักงานหน้างาน — ยังทำไม่ได้ในระดับนี้
สิ่งที่ทำได้แบบซื่อสัตย์คือทดลองเชิงข้อมูล (data-driven) ว่า "การรู้ค่า Q ของ HX ตัวก่อนหน้า ช่วยพยากรณ์อะไรต่อได้ไหม"

In [28]:
chain = {
    'E101AB': None, 'E101CD': None, 'E101EF': None, 'E102': 'E101AB',
    'E103AB': None, 'E104': 'E103AB', 'E105AB': 'E104',
    'E106AB': None, 'E107AB': 'E106AB', 'E108AB': 'E107AB', 'E109AB': 'E108AB',
    'E110ABC': None, 'E111': 'E110ABC', 'E112AB': 'E111', 'E113A': 'E112AB',
}
fig, ax = plt.subplots(figsize=(10, 6))
levels = {}
def depth(hx):
    if hx not in levels:
        levels[hx] = 0 if chain[hx] is None else depth(chain[hx]) + 1
    return levels[hx]
for hx in chain:
    depth(hx)
lanes = {}
for hx, d in levels.items():
    lanes.setdefault(d, []).append(hx)
for d, hxs in lanes.items():
    for i, hx in enumerate(hxs):
        ax.scatter(d, i, s=800, color='#4c72b0')
        ax.annotate(hx, (d, i), ha='center', va='center', color='white', fontsize=8)
        if chain[hx] is not None:
            pd_ = levels[chain[hx]]
            pi = lanes[pd_].index(chain[hx])
            ax.plot([pd_, d], [pi, i], color='grey', lw=1, zorder=0)
ax.set_xlabel('ลำดับขั้น (ใกล้เตามากขึ้นทางขวา)')
ax.set_yticks([])
ax.set_title('ลำดับการไหลของ crude ผ่าน HX แต่ละตัว (topology เท่านั้น ไม่ใช่โมเดล cascade)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_network_topology.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:29: UserWarning: Glyph 3621 (\N{THAI CHARACTER LO LING}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:29: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:29: UserWarning: Glyph 3604 (\N{THAI CHARACTER DO DEK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:29: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:29: UserWarning: Glyph 3610 (\N{THAI CHARACTER BO BAIMAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:29: UserWarning: Glyph 3586 (\N{THAI CHARACTER KHO KHAI}

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:30: UserWarning: Glyph 3638 (\N{THAI CHARACTER SARA UE}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '06_network_topology.png'), dpi=110)


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\848531840.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### การทดลอง: เพิ่ม feature ของ HX ต้นน้ำ (upstream) ช่วยพยากรณ์ CIT ของ HX ปลายน้ำได้ไหม?

ทดลองแบบเจาะจง: ใช้ `E112AB` เป็น target ทดสอบ (HX ก่อน E113A ตัวสุดท้าย) เทียบโมเดลที่มี vs ไม่มี
feature ของ `E111` (upstream) แบบ lag-1 วัน — เพื่อดูว่า "รู้ค่า upstream ล่วงหน้า" ช่วยพยากรณ์ Q_norm
ของ HX ถัดไปได้จริงหรือไม่ (การทดลองนี้เป็นแค่ตัวอย่างสำรวจ ไม่ใช่ผลสรุปสุดท้าย)

In [29]:
target_hx, upstream_hx = 'E112AB', 'E111'
local = pd.DataFrame(index=proc_clean.index)
local['y'] = hx_results[target_hx]['Q_norm']
for lag in [1, 2, 3]:
    local[f'y_lag{lag}'] = local['y'].shift(lag)
local_with_upstream = local.copy()
local_with_upstream['upstream_Q_norm_lag1'] = hx_results[upstream_hx]['Q_norm'].shift(1)

for label, table in [('ไม่มี upstream', local), ('มี upstream', local_with_upstream)]:
    d = table.dropna()
    split_i = int(len(d) * 0.8)
    cols = [c for c in d.columns if c != 'y']
    Xtr, Xte = d[cols].iloc[:split_i], d[cols].iloc[split_i:]
    ytr, yte = d['y'].iloc[:split_i], d['y'].iloc[split_i:]
    model = LinearRegression().fit(Xtr, ytr)
    r2 = r2_score(yte, model.predict(Xte))
    print(f'{label}: R2 = {r2:.4f}  (n_features={len(cols)}, n_test={len(yte)})')

ไม่มี upstream: R2 = 0.9747  (n_features=3, n_test=377)
มี upstream: R2 = 0.9728  (n_features=4, n_test=377)


**⚠️ ผลข้างบน (E112AB←E111 เท่านั้น) ยังตอบคำถาม cascade ไม่ได้จริง:** R² แทบไม่ต่างกัน
(0.9747 vs 0.9728) เพราะ `y_lag1/2/3` (autoregressive ของตัวเองวันก่อนหน้า) น่าจะอธิบาย variance
เกือบทั้งหมดไปแล้ว ทำให้แยกไม่ออกว่า "upstream ไม่มีผลจริง" หรือ "lag-1 ของตัวเองกลบสัญญาณ upstream ไปหมด"
และเป็นการทดสอบแค่คู่เดียวจาก chain 15 ตัว ไม่ใช่คำตอบของคำถามที่ตั้งไว้ตอนต้น ("ถ้า Q ตัวนี้ลด
ตัวไหนจะลดตาม ส่งผลต่อ CIT แบบไหน") — ทำ 2 อย่างเพิ่มเพื่อให้เห็นภาพชัดขึ้น:
1. **Lagged cross-correlation** ก่อน ว่า upstream นำ (lead) downstream จริงไหมที่ lag ไหน ก่อนเชื่อผล regression
2. **ทดสอบซ้ำกับ edge อื่นในสาย** ไม่ใช่แค่คู่เดียว

In [30]:
edges_to_test = [('E112AB', 'E111'), ('E111', 'E110ABC'), ('E102', 'E101AB'), ('E109AB', 'E108AB')]

fig, axes = plt.subplots(1, len(edges_to_test), figsize=(4 * len(edges_to_test), 3.5), sharey=True)
for ax, (down, up) in zip(axes, edges_to_test):
    a = hx_results[down]['Q_norm']
    b = hx_results[up]['Q_norm']
    both = pd.concat([a, b], axis=1, keys=['down', 'up']).dropna()
    lags = range(-7, 8)
    xcorr = [both['up'].shift(lag).corr(both['down']) for lag in lags]
    ax.stem(list(lags), xcorr)
    ax.axvline(0, color='grey', lw=0.7)
    ax.set_title(f'{down} vs {up}\n(lag>0 = upstream นำ)', fontsize=9)
    ax.set_xlabel('lag (วัน)')
axes[0].set_ylabel('correlation')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_cross_correlation_by_edge.png'), dpi=110)
plt.show()

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:15: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:15: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:15: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:15: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:16: UserWarning: Glyph 3609 (\N{THAI CHARACTER NO NU}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '06_cross_correlation_by_edge.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:16: UserWarning: Glyph 3635 (\N{THAI CHARACTER SARA AM}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '06_cross_correlation_by_edge.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:16: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '06_cross_correlation_by_edge.png'), dpi=110)
C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:16: UserWarning: Glyph 3633 (\N{THAI CHARACTER MAI HAN-AKAT}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(FIG_DIR, '06_cross_correlation_by_edge.png'), dpi=110)


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\2455886491.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
edge_results = []
for down, up in edges_to_test:
    local = pd.DataFrame(index=proc_clean.index)
    local['y'] = hx_results[down]['Q_norm']
    for lag in [1, 2, 3]:
        local[f'y_lag{lag}'] = local['y'].shift(lag)
    local_up = local.copy()
    local_up['upstream_Q_norm_lag1'] = hx_results[up]['Q_norm'].shift(1)

    row = {'downstream': down, 'upstream': up}
    for label, table in [('r2_no_upstream', local), ('r2_with_upstream', local_up)]:
        d = table.dropna()
        split_i = int(len(d) * 0.8)
        cols = [c for c in d.columns if c != 'y']
        Xtr, Xte = d[cols].iloc[:split_i], d[cols].iloc[split_i:]
        ytr, yte = d['y'].iloc[:split_i], d['y'].iloc[split_i:]
        model = LinearRegression().fit(Xtr, ytr)
        row[label] = r2_score(yte, model.predict(Xte))
    row['delta_r2'] = row['r2_with_upstream'] - row['r2_no_upstream']
    edge_results.append(row)

edge_df = pd.DataFrame(edge_results).set_index(['downstream', 'upstream'])
edge_df

,,r2_no_upstream,r2_with_upstream,delta_r2
downstream,upstream,,,
E112AB,E111,0.974659,0.972836,-0.001823
E111,E110ABC,0.865257,0.874509,0.009252
E102,E101AB,0.899662,0.914419,0.014757
E109AB,E108AB,0.716586,0.716593,0.000007


## Section 7 — Engineering Data Acceptance EDA (canonical cross-check)

ส่วนนี้เป็น **acceptance gate ก่อนนำข้อมูลไปวิเคราะห์หรือสร้างโมเดล** และให้ถือเป็นข้อสรุปที่เข้มกว่าส่วน exploratory ก่อนหน้า โดยอ่าน topology, operating state และสูตรจาก production source-of-truth แทนการเดาใหม่ใน notebook

### ลำดับพิจารณา
1. **Provenance & alignment** — เวลา, duplicate, coverage และไฟล์ที่ join กันตรงหรือไม่
2. **Sensor quality** — missing, frozen/stale, jump และช่วงค่าทางกายภาพ
3. **Operating context** — shutdown, OFF, NORMAL, SUBSTITUTE_ACTIVE ต้องไม่ถูกปนกัน
4. **Physics** — ตรวจ $\Delta T=T_{out}-T_{in}$, flow split และสูตร $Q=\rho F C_p\Delta T/3600$
5. **Network** — รวม Q ตาม branch; E113A/E112C เลือกตัวที่ active และห้ามบวกซ้ำ
6. **Relationship** — ดูทั้ง level, first difference และ multivariable model พร้อม control variables
7. **Acceptance decision** — วันที่ผิดเงื่อนไขต้อง flag เพื่อ investigate; ไม่ replace ค่าเงียบๆ

> ข้อจำกัด: `Q_norm = Q / total charge` เป็น normalized cold-side heat recovery ไม่ใช่ U หรือ fouling resistance โดยตรง และ Q ของ final position ใช้ CIT เป็น cold-out จึงเกิด target leakage ถ้าเอา same-day final Q ไปทำนาย CIT.

In [32]:
# 7.1 Load canonical inputs and plant topology (standalone cell)
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'domain' / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'domain' / 'config.py').exists():
    raise FileNotFoundError('Cannot locate project root containing src/domain/config.py')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.domain.config import (HX_CONFIG as HX_CANON, CIT_TAG as CIT_CANON,
                               TOTAL_CHARGE_TAG as CHARGE_CANON, DATA_DIR as DATA_CANON)
from src.features.crude_properties import cp_rho_crude

EDA_FIG_DIR = PROJECT_ROOT / 'notebooks' / 'eda' / 'figures' / 'engineering_acceptance'
EDA_FIG_DIR.mkdir(parents=True, exist_ok=True)

def read_ts_csv(name):
    x = pd.read_csv(Path(DATA_CANON) / name, parse_dates=['Timestamp'])
    return x.set_index('Timestamp').sort_index()

eda_proc = read_ts_csv('Process_information_with_crude.csv')
eda_state = read_ts_csv('Operating_State.csv')
eda_q = read_ts_csv('Feature_Q.csv')
eda_feat = read_ts_csv('Feature_calculated.csv')

common_idx = eda_proc.index.intersection(eda_state.index).intersection(eda_q.index)
eda_proc, eda_state, eda_q = [x.reindex(common_idx) for x in (eda_proc, eda_state, eda_q)]
print(f'Canonical aligned period: {common_idx.min().date()} -> {common_idx.max().date()} | n={len(common_idx):,}')
print('Duplicate timestamps:', {'process': int(eda_proc.index.duplicated().sum()),
                                'state': int(eda_state.index.duplicated().sum()),
                                'Q': int(eda_q.index.duplicated().sum())})

Canonical aligned period: 2021-01-01 -> 2026-07-01 | n=1,898
Duplicate timestamps: {'process': 0, 'state': 0, 'Q': 0}


In [33]:
# 7.2 Coverage, missingness and frozen-sensor screening
required_tags = sorted({CIT_CANON, CHARGE_CANON} |
                       {v[k] for v in HX_CANON.values() for k in ('cold_flow', 'cold_in', 'cold_out')})
tag_frame = eda_proc[required_tags]

def longest_true_run(mask):
    groups = mask.ne(mask.shift()).cumsum()
    return int(mask.groupby(groups).sum().max()) if len(mask) else 0

quality_rows = []
for tag in required_tags:
    s = tag_frame[tag]
    frozen = s.diff().abs().lt(1e-9) & s.notna()
    quality_rows.append({
        'tag': tag, 'missing_pct': 100*s.isna().mean(), 'n_unique': s.nunique(dropna=True),
        'frozen_pct': 100*frozen.mean(), 'longest_frozen_days': longest_true_run(frozen),
        'p01': s.quantile(.01), 'median': s.median(), 'p99': s.quantile(.99)})
quality_table = pd.DataFrame(quality_rows).set_index('tag').sort_values(['missing_pct','longest_frozen_days'], ascending=False)
display(quality_table.round(2))

fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)
sns.heatmap(tag_frame.isna().T, cmap=['#f2f2f2','#c44e52'], cbar=False, ax=axes[0])
axes[0].set_title('Required-tag missingness (red = missing)')
axes[0].set_xlabel('Time index'); axes[0].set_ylabel('Tag')
quality_table[['missing_pct','frozen_pct']].sort_values('missing_pct').plot.barh(ax=axes[1])
axes[1].set_title('Missing and frozen fraction by required tag'); axes[1].set_xlabel('% of aligned days')
fig.savefig(EDA_FIG_DIR / '01_tag_quality.png', dpi=130, bbox_inches='tight')
plt.show()

,missing_pct,n_unique,frozen_pct,longest_frozen_days,p01,median,p99
tag,,,,,,,
1TI225.pv,0.0,1156,39.09,742,35.91,85.10,97.97
1FI009.pv,0.0,1713,9.22,164,0.00,167.40,210.00
1TI115.pv,0.0,1897,0.05,1,37.47,219.62,244.16
1FI007.pv,0.0,1898,0.00,0,132.92,192.68,214.83
1FI008.pv,0.0,1898,0.00,0,107.49,155.28,173.53
1FI015.pv,0.0,1898,0.00,0,156.21,218.21,264.04
1FI016.pv,0.0,1898,0.00,0,107.19,141.06,164.72
1FI017.pv,0.0,1898,0.00,0,112.90,162.07,179.59
1TI101.pv,0.0,1898,0.00,0,84.05,90.58,106.04


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\485534926.py:21: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\485534926.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [34]:
# 7.3 Operating-state separation, Delta-T physics and CPHT-2 flow split
valid_states = {'NORMAL', 'SUBSTITUTE_ACTIVE'}
dt_active = pd.DataFrame(index=common_idx)
for hx, cfg in HX_CANON.items():
    active = eda_state[hx].isin(valid_states)
    dt_active[hx] = (eda_proc[cfg['cold_out']] - eda_proc[cfg['cold_in']]).where(active)

state_pct = (eda_state.apply(lambda s: s.value_counts(normalize=True)).fillna(0) * 100).T
dt_summary = pd.DataFrame({
    'active_days': dt_active.notna().sum(),
    'median_dT_C': dt_active.median(),
    'p01_dT_C': dt_active.quantile(.01),
    'nonpositive_pct': 100*(dt_active <= 0).sum()/dt_active.notna().sum().replace(0, np.nan)
}).sort_values('nonpositive_pct', ascending=False)
display(state_pct.round(1)); display(dt_summary.round(2))

branch_flow = eda_proc[['1FI015.pv','1FI016.pv','1FI017.pv']].sum(axis=1, min_count=3)
flow_ratio = branch_flow / eda_proc[CHARGE_CANON].replace(0, np.nan)
running = eda_proc[CHARGE_CANON] >= 200

fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)
sns.boxplot(data=dt_active, orient='h', showfliers=False, ax=axes[0])
axes[0].axvline(0, color='red', ls='--'); axes[0].set_title('Active-state cold-side Delta-T distribution')
axes[0].set_xlabel('Tout - Tin (degC)')
flow_ratio.where(running).plot(ax=axes[1], lw=.8, label='(FI015+FI016+FI017) / total charge')
axes[1].axhline(1, color='black', lw=1); axes[1].axhspan(.85, 1.15, color='green', alpha=.12, label='screening band')
axes[1].set_title('CPHT-2 branch-flow closure screening'); axes[1].set_ylabel('flow ratio'); axes[1].legend()
fig.savefig(EDA_FIG_DIR / '02_state_physics_flow.png', dpi=130, bbox_inches='tight')
plt.show()

,BYPASS_CANDIDATE,BYPASS_UPSTREAM_ACTIVE,NORMAL,OFF,SUBSTITUTED,SUBSTITUTE_ACTIVE,UNMEASURED_ACTIVE
E101AB,0.0,0.0,100.0,0.0,0.0,0.0,0.0
E101CD,0.0,0.0,100.0,0.0,0.0,0.0,0.0
E101EF,0.0,0.0,91.3,0.0,8.7,0.0,0.0
E101G,0.0,0.0,0.0,91.3,0.0,0.0,8.7
E102,0.0,0.0,100.0,0.0,0.0,0.0,0.0
E113A,0.0,0.1,90.0,0.0,9.9,0.0,0.0
E112C,0.0,0.1,0.0,79.6,0.0,20.3,0.0
E112AB,0.1,0.0,99.9,0.0,0.0,0.0,0.0
E106AB,0.0,0.0,100.0,0.0,0.0,0.0,0.0
E110ABC,0.0,0.0,100.0,0.0,0.0,0.0,0.0


,active_days,median_dT_C,p01_dT_C,nonpositive_pct
E108AB,1898,19.48,-37.09,2.58
E101EF,1733,41.22,-0.14,1.10
E111,1898,24.72,6.89,0.26
E107AB,1898,30.17,10.81,0.11
E101AB,1898,53.61,46.67,0.00
E101CD,1898,53.69,47.03,0.00
E102,1898,5.58,3.72,0.00
E103AB,1898,45.43,27.69,0.00
E104,1898,37.12,29.98,0.00
E105AB,1898,62.42,47.59,0.00


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\376087394.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [35]:
# 7.4 Independent equation replay: Q = rho * flow * Cp * DeltaT / 3600; Q_norm = Q / total charge
q_recalc = pd.DataFrame(index=common_idx)
equation_rows = []
sg = eda_proc['SG_15_6C']
charge = eda_proc[CHARGE_CANON].replace(0, np.nan)
for hx, cfg in HX_CANON.items():
    tin, tout, flow_hx = eda_proc[cfg['cold_in']], eda_proc[cfg['cold_out']], eda_proc[cfg['cold_flow']]
    cp, rho = cp_rho_crude((tin+tout)/2, sg)
    q_recalc[hx] = rho * flow_hx * cp * (tout-tin) / 3600 / charge
    active = eda_state[hx].isin(valid_states) & eda_q[hx].notna()
    err = (q_recalc[hx] - eda_q[hx]).where(active)
    rel = (err.abs() / eda_q[hx].abs().clip(lower=.1)).where(active)
    equation_rows.append({'HX': hx, 'n_compared': int(err.notna().sum()),
                          'median_abs_error': err.abs().median(), 'p95_relative_error_pct': 100*rel.quantile(.95)})
equation_audit = pd.DataFrame(equation_rows).set_index('HX').sort_values('p95_relative_error_pct', ascending=False)
display(equation_audit.round(4))

fig, axes = plt.subplots(4, 4, figsize=(15, 13), constrained_layout=True)
for ax, hx in zip(axes.flat, HX_CANON):
    d = pd.concat([eda_q[hx], q_recalc[hx]], axis=1, keys=['stored','recalculated']).dropna()
    ax.scatter(d['stored'], d['recalculated'], s=5, alpha=.25)
    lo, hi = d.min().min(), d.max().max(); ax.plot([lo,hi],[lo,hi], color='red', lw=1)
    ax.set_title(hx); ax.set_xlabel('stored'); ax.set_ylabel('recalculated')
for ax in axes.flat[len(HX_CANON):]: ax.axis('off')
fig.suptitle('Cold-side Q_norm equation replay (1:1 line)', y=1.02)
fig.savefig(EDA_FIG_DIR / '03_q_equation_replay.png', dpi=130, bbox_inches='tight')
plt.show()

,n_compared,median_abs_error,p95_relative_error_pct
HX,,,
E112C,385,0.0,0.0
E112AB,1896,0.0,0.0
E105AB,1898,0.0,0.0
E101EF,1697,0.0,0.0
E113A,1708,0.0,0.0
E106AB,1898,0.0,0.0
E110ABC,1898,0.0,0.0
E108AB,1841,0.0,0.0
E107AB,1895,0.0,0.0


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\1202251547.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
# 7.5 Network aggregation and level-vs-change relationship with CIT
branches = {
    'Branch A': ['E103AB','E104','E105AB'],
    'Branch B': ['E106AB','E107AB','E108AB','E109AB'],
    'Branch C': ['E110ABC','E111','E112AB'],
}
network_q = pd.DataFrame(index=common_idx)
for name, hxs in branches.items():
    network_q[name] = eda_q[hxs].sum(axis=1, min_count=len(hxs))
# E113A and E112C are alternate shells at one measured position: choose active, never sum.
network_q['Final active'] = eda_q['E113A'].where(eda_state['E113A'].isin(valid_states)).combine_first(
                            eda_q['E112C'].where(eda_state['E112C'].isin(valid_states)))
network_q['CPHT-2 total'] = network_q[['Branch A','Branch B','Branch C','Final active']].sum(axis=1, min_count=4)
cit_eda = eda_proc[CIT_CANON]

relation_rows = []
fig, axes = plt.subplots(2, 5, figsize=(18, 7), constrained_layout=True)
for j, col in enumerate(network_q.columns):
    level = pd.concat([network_q[col], cit_eda], axis=1, keys=['Q','CIT']).dropna()
    change = level.diff().dropna()
    for i, (d, label) in enumerate([(level, 'level'), (change, 'day-to-day change')]):
        ax = axes[i,j]; ax.scatter(d['Q'], d['CIT'], s=5, alpha=.25)
        if len(d) > 2:
            slope, intercept = np.polyfit(d['Q'], d['CIT'], 1)
            xx = np.array([d['Q'].quantile(.01), d['Q'].quantile(.99)])
            ax.plot(xx, intercept+slope*xx, color='red')
            r = d.corr().iloc[0,1]
            relation_rows.append({'network_feature': col, 'view': label, 'r': r, 'univariate_slope': slope, 'n': len(d)})
            ax.set_title(f'{col} | {label}\nr={r:.2f}, slope={slope:.2f}')
        ax.set_xlabel('Q_norm' if i==0 else 'dQ_norm/day'); ax.set_ylabel('CIT' if i==0 else 'dCIT/day')
fig.savefig(EDA_FIG_DIR / '04_network_level_vs_change.png', dpi=130, bbox_inches='tight')
plt.show()
relation_table = pd.DataFrame(relation_rows)
display(relation_table.round(3))

C:\Users\saard\AppData\Local\Temp\ipykernel_51336\246811452.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,network_feature,view,r,univariate_slope,n
0,Branch A,level,0.274,0.327,1898
1,Branch A,day-to-day change,0.245,0.391,1897
2,Branch B,level,-0.029,-0.043,1838
3,Branch B,day-to-day change,0.147,0.289,1837
4,Branch C,level,0.141,0.163,1889
5,Branch C,day-to-day change,0.223,0.415,1888
6,Final active,level,0.034,0.008,1896
7,Final active,day-to-day change,0.012,0.003,1895
8,CPHT-2 total,level,0.150,0.028,1829
9,CPHT-2 total,day-to-day change,0.099,0.024,1828


In [37]:
# 7.6 Simultaneous rate model: dCIT = sum(W_i*dQ_i) + controls + error
# Deliberately exclude same-day Final active Q because its cold_out is CIT (target leakage).
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

rate = network_q[['Branch A','Branch B','Branch C']].diff().add_prefix('dQ_')
rate['dT_CPHT2_in'] = eda_proc['1TI225.pv'].diff()
rate['dCharge'] = eda_proc[CHARGE_CANON].diff()
rate['dSG'] = eda_proc['SG_15_6C'].diff()
rate['dCIT'] = cit_eda.diff()
rate = rate.replace([np.inf,-np.inf], np.nan).dropna()
# Robust screening only; flagged points remain visible in Section 7.7 and are not silently imputed.
rate_fit = rate[rate['dCIT'].abs() <= rate['dCIT'].abs().quantile(.99)].copy()
X_rate, y_rate = rate_fit.drop(columns='dCIT'), rate_fit['dCIT']
split = int(.8*len(rate_fit)); Xtr, Xte = X_rate.iloc[:split], X_rate.iloc[split:]
ytr, yte = y_rate.iloc[:split], y_rate.iloc[split:]
scaler = StandardScaler().fit(Xtr)
model = RidgeCV(alphas=np.logspace(-3,3,31), cv=TimeSeriesSplit(5)).fit(scaler.transform(Xtr), ytr)
pred = model.predict(scaler.transform(Xte))
coef_native = pd.Series(model.coef_/scaler.scale_, index=X_rate.columns, name='coefficient_native_units')
coef_std = pd.Series(model.coef_, index=X_rate.columns, name='coefficient_per_1SD')
coef_table = pd.concat([coef_native, coef_std], axis=1).sort_values('coefficient_per_1SD', key=abs, ascending=False)
display(coef_table.round(4))
print({'alpha': model.alpha_, 'test_MAE_degC_per_day': round(mean_absolute_error(yte,pred),3),
       'baseline_zero_change_MAE': round(mean_absolute_error(yte,np.zeros(len(yte))),3)})

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
coef_std.sort_values().plot.barh(ax=axes[0]); axes[0].axvline(0,color='black',lw=.8)
axes[0].set_title('Rate-model coefficient per 1 SD'); axes[0].set_xlabel('effect on dCIT (degC/day)')
axes[1].scatter(yte, pred, s=8, alpha=.4)
lims=[min(yte.min(),pred.min()),max(yte.max(),pred.max())]; axes[1].plot(lims,lims,color='red')
axes[1].set(xlabel='measured dCIT', ylabel='predicted dCIT', title='Chronological holdout: measured vs predicted')
fig.savefig(EDA_FIG_DIR / '05_rate_model.png', dpi=130, bbox_inches='tight')
plt.show()

,coefficient_native_units,coefficient_per_1SD
dQ_Branch C,0.6986,0.5201
dQ_Branch A,0.5306,0.4822
dQ_Branch B,0.6192,0.4781
dT_CPHT2_in,0.2323,0.2111
dSG,-34.9442,-0.1504
dCharge,0.0006,0.0063


{'alpha': np.float64(158.48931924611142), 'test_MAE_degC_per_day': 0.528, 'baseline_zero_change_MAE': 0.691}


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\38721752.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
# 7.7 Date-level anomaly gate and explicit acceptance summary
anomaly = pd.DataFrame(index=common_idx)
anomaly['missing_required_tag'] = tag_frame.isna().any(axis=1)
anomaly['shutdown_or_low_charge'] = eda_proc[CHARGE_CANON] < 200
anomaly['flow_closure_outside_15pct'] = running & ~flow_ratio.between(.85,1.15)
anomaly['any_active_nonpositive_dT'] = (dt_active <= 0).any(axis=1)
anomaly['large_CIT_jump'] = cit_eda.diff().abs() > 5
anomaly['final_state_conflict'] = (eda_state['E113A'].isin(valid_states) == eda_state['E112C'].isin(valid_states))
anomaly['n_flags'] = anomaly.sum(axis=1)

acceptance = pd.DataFrame({
    'check': ['timestamps unique','required-tag completeness','active dT positive','branch-flow closure','final shell state exclusive','Q equation replay'],
    'observed': [str(not eda_proc.index.duplicated().any()),
                 f'{100*(~anomaly.missing_required_tag).mean():.1f}% complete days',
                 f'{100*(~anomaly.any_active_nonpositive_dT).mean():.1f}% days pass',
                 f'{100*(~anomaly.flow_closure_outside_15pct).mean():.1f}% days pass',
                 f'{100*(~anomaly.final_state_conflict).mean():.1f}% days pass',
                 f'max p95 rel err={equation_audit.p95_relative_error_pct.max():.3f}%'],
    'action_if_fail': ['deduplicate before join','trace source tag/gap; do not blind-fill',
                       'check state/tag direction/sensor','check meters, bypass and split topology',
                       'exclude date and review switch logic','stop and reconcile units/formula/version']
})
display(acceptance); display(anomaly.sum().sort_values(ascending=False).to_frame('flagged_days'))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
anomaly.drop(columns='n_flags').sum().sort_values().plot.barh(ax=axes[0])
axes[0].set_title('Flagged days by anomaly rule'); axes[0].set_xlabel('days')
anomaly['n_flags'].plot(ax=axes[1], lw=.8); axes[1].set_title('Anomaly count through time')
axes[1].set_ylabel('number of flags')
fig.savefig(EDA_FIG_DIR / '06_anomaly_gate.png', dpi=130, bbox_inches='tight')
plt.show()

print('Use rule: model-ready rows require n_flags == 0 plus an HX-specific active state. Keep flagged rows for root-cause review, not automatic replacement.')

,check,observed,action_if_fail
0,timestamps unique,True,deduplicate before join
1,required-tag completeness,100.0% complete days,trace source tag/gap; do not blind-fill
2,active dT positive,96.2% days pass,check state/tag direction/sensor
3,branch-flow closure,100.0% days pass,"check meters, bypass and split topology"
4,final shell state exclusive,89.5% days pass,exclude date and review switch logic
5,Q equation replay,max p95 rel err=0.000%,stop and reconcile units/formula/version


,flagged_days
n_flags,282
final_state_conflict,199
any_active_nonpositive_dT,73
large_CIT_jump,10
missing_required_tag,0
shutdown_or_low_charge,0
flow_closure_outside_15pct,0


Use rule: model-ready rows require n_flags == 0 plus an HX-specific active state. Keep flagged rows for root-cause review, not automatic replacement.


C:\Users\saard\AppData\Local\Temp\ipykernel_51336\500871711.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### วิธีอ่านผล Section 7

- ถ้า equation replay ไม่อยู่ใกล้เส้น 1:1: หยุดก่อน เพราะสูตร/หน่วย/version ไม่ตรงกัน
- ถ้า level correlation สูง แต่ change correlation ต่ำ: ส่วนใหญ่เป็น regime/ฤดูกาล/throughput ไม่ควรเรียกว่า causal effect
- coefficient ใน rate model เป็นผลพร้อมกันหลังควบคุมตัวแปรที่ใส่แล้ว ไม่ใช่ universal physical constant และต้องเทียบ holdout กับ baseline
- ห้ามใช้ same-day E113A/E112C Q เป็น feature ทำนาย CIT เพราะ Q นั้นคำนวณจาก CIT เอง
- สำหรับ fouling ให้พิจารณา U, Rf, U_relative ภายใน run/operating state และ cleaning validation; ไม่สรุปจาก Q_norm trend ตัวเดียว
- ตาราง anomaly คือรายการวันสำหรับ root-cause investigation: sensor, switching, bypass, shutdown, crude change หรือเหตุการณ์ cleaning ต้องถูกแยกก่อนนำไป train model

## สรุปและขั้นต่อไป

- ทุกสูตรและ threshold ใน notebook นี้เขียนใหม่เอง สามารถปรับ (`SHUTDOWN_THRESH`, `jump_frac`, z-score window ฯลฯ) ได้ตรงๆ ในเซลล์ที่เกี่ยวข้อง
- Section 3-4 ใช้ Q_norm เป็น proxy ของ U จริง — ถ้าต้องการค่า U ที่แม่นยำต้องหาข้อมูล Area/LMTD ของแต่ละ HX มาเพิ่ม
- Fouling rate (Section 3) และ cleaning-event gain (Section 3) เป็น **INFERRED แบบง่าย ยังไม่ยืนยัน** — ดูคำเตือนในเซลล์ที่เกี่ยวข้อง อย่าถือเป็นตัวเลขทางการ
- Finding ที่เจอเอง (persistence baseline แข่งกับ ML ในการพยากรณ์ CIT) ตรงกับสิ่งที่เคยพบใน pipeline เดิม — เป็น sanity check ที่ดีว่าไม่ได้เข้าใจผิดอะไรไป
- **Network section (6.2) เป็นการสำรวจสัญญาณเบื้องต้น (exploratory signal check) เท่านั้น ไม่ใช่โมเดล cascade/what-if ที่ผ่านการยืนยัน** —
  ทดสอบได้แค่ 4 edge จาก chain 15 ตัว, cross-correlation ที่ดูก็ยังไม่ควบคุม confounder ร่วม (เช่น total charge เปลี่ยนพร้อมกันทั้งสาย
  ทำให้ HX ทุกตัวขยับพร้อมกันโดยไม่ต้องมี "cascade" จริง) ถ้าจะทำโมเดล cascade ที่ตอบคำถาม "Q ตัวนี้ลด ตัวไหนลดตาม ส่งผลต่อ CIT แค่ไหน"
  ได้จริง ต้องมี: ทดสอบครบทั้ง 15 node, feature ที่แยก confounder ร่วม (เช่น de-trend ด้วย total charge ก่อน), และคุยกับวิศวกรหน้างาน
  เพื่อยืนยัน topology ที่ใช้จริง — ยังไม่ได้ทำในระดับนี้